# UKCP18 CPM to HEC-RAS DSS

任意时间段与 metadata 网格适配。修改下面 CONFIG 后先检查 metadata，再执行。

先准备 payload；正式写 DSS 时设置 PREPARE_ONLY=False 并填写 Jython/template。

In [ ]:
from pathlib import Path
import json
from dss_pipeline import main


## 配置 / Configuration

`window` 可任意增删；日期结束日包含当天，带时分的结束时间为不包含。
原生旋转网格需增加 `target_crs="EPSG:27700"` 与 `target_cell_size=2200`。

In [ ]:
CONFIG = {'input_dir': 'D:/climate/cpm', 'input_pattern': 'pr*.nc', 'output_dir': 'D:/output/cpm', 'variable': 'pr', 'window': [['01', '2019-11-01', '2039-12-31'], ['02', '2039-11-01', '2059-12-31'], ['03', '2059-11-01', '2080-11-30']], 'prepare_only': True, 'buffer_cells': 1, 'chunk_records': 24, 'missing_data': 'zero', 'resampling': 'average'}

RUN = False
PREPARE_ONLY = True
# CONFIG["jython_exe"] = "D:/HEC-DSSVue/Jython.exe"
# CONFIG["template_dss"] = "examples/output_dss/pr_forth_20y_01_20191101_20391231.dss"


In [ ]:
def cli_arguments(config):
    arguments = []
    for key, value in config.items():
        flag = "--" + key.replace("_", "-")
        if key == "window":
            for row in value:
                arguments.extend([flag, *map(str, row)])
        elif key == "select":
            for item in value:
                arguments.extend([flag, str(item)])
        elif isinstance(value, bool):
            if value:
                arguments.append(flag)
        elif isinstance(value, list):
            arguments.extend([flag, *map(str, value)])
        elif value is not None:
            arguments.extend([flag, str(value)])
    return arguments


## 先检查 / Inspect

把 CHECK_METADATA 改为 True 后，读取真实文件的坐标、日历、单位与可用日期；不写输出。

In [ ]:
CHECK_METADATA = False
if CHECK_METADATA:
    inspection = {**CONFIG, "inspect": True}
    main('cpm', cli_arguments(inspection), clip=False)


## 执行 / Run

设 RUN=True。所有源文件只读。已有 DSS 需要显式 overwrite=True。

In [ ]:
if RUN:
    settings = {**CONFIG, "prepare_only": PREPARE_ONLY}
    jobs = main('cpm', cli_arguments(settings), clip=False)
    for meta_path in jobs:
        metadata = json.loads(Path(meta_path).read_text(encoding="ascii"))
        print(metadata["start"], metadata["end_exclusive"], metadata["record_count"], metadata["cell_size"])
else:
    print("Edit CONFIG, inspect metadata, then set RUN=True.")
